# 1. Configurando o Ambiente e a Base de Dados
Nesta etapa nós prepararemos a configuração inicial do Algoritmo Evolucionário e do projeto.

In [2]:
# Importando as bibliotecas essenciais e carregando os dados
import pandas as pd
import numpy as np
import sys
import os

# Adicionando a pasta src para importar a engine GA que criamos
sys.path.append(os.path.abspath('../src'))
from ga_optimizer import GeneticAlgorithmOptimizer

# Lendo os dados já preprocessados e escalonados
X_train = pd.read_csv('../data/processed/X_train_scaled.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze() # transforma coluna em series

print(f"Base de treino carregada: {X_train.shape[0]} amostras, {X_train.shape[1]} features.")

Base de treino carregada: 455 amostras, 30 features.


# 2. Carregando os dados de Treinamento
Lendo os dados de rastreio processados.

In [3]:
import time

print("Iniciando Experimento 1...")
start_time = time.time()

# Instanciando o otimizador
ga_exp1 = GeneticAlgorithmOptimizer(
    X_train=X_train,
    y_train=y_train,
    population_size=10,
    generations=5,
    mutation_rate=0.1
)

# Rodando o algoritmo
best_params_exp1 = ga_exp1.run()

end_time = time.time()
print(f"Tempo de execução (Experimento 1): {end_time - start_time:.2f} segundos")

Iniciando Experimento 1...
Iniciando Otimização GA (5 Gerações | População: 10)
--- Geração 1 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 2 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 3 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 4 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 5 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)

=== Fim da Otimização ===
Melhor Conjunto de Hiperparâmetros: {'n_estimators': 300, 'max_depth': 10, 'min_samples_split': 10, 'class_weight': 'balanced_subsample'}
Melhor Score Fitness: 0.9493
Tempo de execução (Experimento 1): 46.79 segundos


Vamos rodar o GA com uma configuração modesta. Como o problema de Câncer de Mama é sensível, queremos ver como o modelo de regressão não-linear (Random Forest) converge rápido para dar luz ao Recall.

In [4]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Construindo o modelo final vencedor
best_rf_model = RandomForestClassifier(**best_params_exp1, random_state=42)
best_rf_model.fit(X_train, y_train)

# Salvando a árvore na pasta models para os proximos testes
output_dir = '../outputs/models/'
os.makedirs(output_dir, exist_ok=True)
joblib.dump(best_rf_model, os.path.join(output_dir, 'best_rf_model_ga.pkl'))

print("Modelo otimizado via Algoritmo Genético salvo com sucesso!")

Modelo otimizado via Algoritmo Genético salvo com sucesso!


## Experimento 2: População Maior (Exploração Média)
Agora vamos aumentar o tamanho da população para 20 e dobrar as gerações para 10. A taxa de mutação continuará em 0.1. A expectativa é encontrar hiperparâmetros ainda melhores com uma busca mais abrangente no espaço.

In [5]:
print("Iniciando Experimento 2...")
start_time_exp2 = time.time()

# Experimento 2: População maior, mais gerações
ga_exp2 = GeneticAlgorithmOptimizer(
    X_train=X_train,
    y_train=y_train,
    population_size=20,
    generations=10,
    mutation_rate=0.1
)

best_params_exp2 = ga_exp2.run()

end_time_exp2 = time.time()
print(f"Tempo de execução (Experimento 2): {end_time_exp2 - start_time_exp2:.2f} segundos")

Iniciando Experimento 2...
Iniciando Otimização GA (10 Gerações | População: 20)
--- Geração 1 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 2 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 3 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 4 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 5 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 6 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 7 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 8 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 9 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)
--- Geração 10 ---
Melhor Fitness da Geração: 0.9493 (Recall: 0.9411, Spec: 0.9684)

=== Fim da Otimização ===
Melhor Conjunto de Hiperparâmetros: {'n_estimators': 300, 'ma

Após a busca pela arquitetura perfeita do Random Forest gerada pela evolução, vamos instanciar o modelo definitivo para poder avaliá-lo mais tarde.

In [6]:
print("Iniciando Experimento 3...")
start_time_exp3 = time.time()

# Experimento 3: Alta mutação
ga_exp3 = GeneticAlgorithmOptimizer(
    X_train=X_train,
    y_train=y_train,
    population_size=15,
    generations=8,
    mutation_rate=0.3
)

best_params_exp3 = ga_exp3.run()

end_time_exp3 = time.time()
print(f"Tempo de execução (Experimento 3): {end_time_exp3 - start_time_exp3:.2f} segundos")

Iniciando Experimento 3...
Iniciando Otimização GA (8 Gerações | População: 15)
--- Geração 1 ---
Melhor Fitness da Geração: 0.9463 (Recall: 0.9353, Spec: 0.9719)
--- Geração 2 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 3 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 4 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 5 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 6 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 7 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)
--- Geração 8 ---
Melhor Fitness da Geração: 0.9473 (Recall: 0.9353, Spec: 0.9754)

=== Fim da Otimização ===
Melhor Conjunto de Hiperparâmetros: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 5, 'class_weight': 'balanced_subsample'}
Melhor Score Fitness: 0.9473
Tempo de execução (Experimento 3): 65.35 segundos
